# ANÁLISIS DE LOS COSTOS DE PROCESO DE PRODUCCIÓN

## Extracción y limpieza de los datos

In [33]:
# Importamos las librerías necesarias para el desarrollo del código
import pandas as pd
import plotly.express as px
import re
import unicodedata

# Cargamos el archivo Excel y leemos la hoja de costos
xls = pd.ExcelFile('jjjjj.xlsx')
df_costos = pd.read_excel('jjjjj.xlsx', sheet_name='Sheet1')

# Cargamos el último archivo de excel para unirlo a este
df_con_comercial = pd.read_excel('df_final.xlsx')

In [34]:
# Conservamos solo las columnas necesarias para el análisis
df_con_comercial = df_con_comercial [[
    'Costeo_Producto',
    'Costeo',
    'Nombre',
    'Nombre del Comercial',
]]

# Unimos las dos tablas para tener el nombre del comercial en el df de costos
df_costos = df_costos.merge(
    df_con_comercial,
    on='Costeo_Producto',
    how='left'
    )

In [35]:
# Definimos la lista de categorías oficiales para la clasificación de productos
categorias_oficiales = [
    'Accesorios', 'Bata', 'Bermuda', 'Blusa', 'Bolígrafo', 'Botas', 'Botella', 'Botón', 'Broche', 'Buzo', 'Camibuzo', 
    'Camiseta', 'Camisa', 'Cachuchera', 'Casco', 'Chaleco', 'Chaqueta', 'Cinta', 'Cofia', 'Conjunto', 'Cordón', 
    'Cremallera', 'Cuello', 'Delantal', 'Diseño', 'Embone', 'Falda', 'Gafas seguridad', 'Gorra', 'Gorro', 'Guantes', 
    'Guata', 'Hilo', 'Hoodie', 'Impermeable', 'Jean', 'Libreta', 'Lonchera', 'Mangas', 'Marquilla', 'Máscara', 'Medias', 
    'Morral', 'Ojalete', 'Overol', 'Pantalón', 'Pantaloneta', 'Pañoleta', 'Paraguas', 'Pavas', 'Polainas', 'Resorte', 
    'Ropa interior', 'Saco', 'Sastre', 'Servicios', 'Sesgo', 'Sudadera', 'Suéter', 'Tankas', 'Tapabocas', 
    'Tapaoídos', 'Tecnología', 'Termo', 'Uniformes', 'Velcro', 'Zapatos', 'Escafandra'
]

# Creamos una función para normalizar el texto, eliminando acentos y convirtiendo a minúsculas
def normalizar(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    return texto

# Creamos una función que clasifique los productos en las categorías oficiales.
def clasificar(Producto):
    producto_norm = normalizar(Producto)
    # Recorremos cada categoría oficial y verificamos si alguna de las palabras está presente en el nombre del producto
    for cat in categorias_oficiales:
        cat_norm = normalizar(cat)
        pattern = r'\b' + re.escape(cat_norm) + r'\b'
        # asignamos la categoría correspondiente si encontramos una coincidencia
        if re.search(pattern, producto_norm):
            return cat
    # Si no encontramos coincidencia, asignamos 'Otros'
    return 'Otros'

df_costos['Categoría'] = df_costos['Producto'].apply(clasificar)   
#print(df_costos['Categoría'].value_counts())
df_costos.columns

Index(['Num-OP', 'Secuencia', 'OP Det', 'Producir', 'Creado El',
       'Costeo_Producto', 'Producto', 'Cliente', 'Cantidad Solicitada', 'Id.',
       'Fecha', 'costo_antes_iva', 'fecha_creacion', 'Proceso Producción',
       'OP-D', 'OS-Valor Unitario', 'OS-Cantidad', 'Num OS',
       'Diferencia valor unitario', 'Valor total OS', 'valor total op',
       'Diferencia', 'BINS', 'Costeo', 'Nombre', 'Nombre del Comercial',
       'Categoría'],
      dtype='object')

## BOXPLOT de la diferencia de costos en cada proceso de producción

In [36]:
# Creamos un gráfico de caja para visualizar la distribución de la diferencia del valor unitario por proceso de producción
df_costos.columns = df_costos.columns.str.strip()

paleta_molt_oscura = ["#29104A", "#4A1B8C", "#0F0530", "#3C1053", "#5C2B92"]
paleta_contraste_oscuro = ["#3C1053", "#003F5C", "#005F4B", "#6B2D3A", "#2F4F4F"]
paleta_textil = ["#5B46DF", "#4E6E58", "#A0522D", "#607B8B", "#8B4513"]

figboxplot = px.box(df_costos, 
             x='Proceso Producción', 
             y='Diferencia valor unitario', 
             points='outliers',
             notched=True,
             color='Proceso Producción',
             hover_data = ['OP Det', 'Nombre del Comercial', 'Costeo_Producto'],
             color_discrete_sequence=paleta_textil)

figboxplot.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
figboxplot.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

figboxplot.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

figboxplot.show()

# BOXPLOT Diferencia de valor total

In [37]:
df_costos.columns = df_costos.columns.str.strip()
figboxtotal = px.box(df_costos, 
             x='Proceso Producción', 
             y='Diferencia', 
             points='outliers',
             notched=True,
             color='Proceso Producción',
             hover_data = ['OP Det', 'Nombre del Comercial', 'Costeo_Producto'],
             color_discrete_sequence=paleta_textil)

figboxtotal.update_layout(
    # 1. Fondo transparente
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    
    # 2. Tipografía Times New Roman
    font=dict(
        family="Times New Roman, Times, serif",
        size=14,
        color="#333333" # Color de letra gris muy oscuro (casi negro)
    ),
    
    # Ajuste opcional de márgenes para que la letra no se corte
    margin=dict(t=80, b=50, l=50, r=50)
)

# 3. Líneas de cuadrícula más oscuras
figboxtotal.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', # Un gris más marcado (puedes usar 'gray' si lo quieres aún más oscuro)
    linecolor='gray',    # Línea del eje principal en negro
    zeroline=True,        # Mantener la línea del cero...
    zerolinecolor='#d3d3d3', # ...pero usar un gris mucho más suave (LightGrey)
    zerolinewidth=1
)

figboxtotal.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='#bdbdbd', 
    linecolor='gray'
)

figboxtotal.show()

## Visualización de datos del valor costeado por categoría

In [38]:
# Agrupamos por Categoría y proceso de producción, calculando la desviación estándar y el promedio del valor costeado
df_calc_pag = df_costos.groupby(['Categoría', 'Proceso Producción'])['costo_antes_iva'].agg(['std', 'mean']).reset_index()

# Calculamos el coeficiente de variación, es decir, el porcentaje de desviación respecto al promedio.
df_calc_pag['inestabilidad'] = df_calc_pag['std'] / df_calc_pag['mean']

# Utilizamos el calculo de inestabilidad para identificar las 5 categorías con procesos más inestables.
top5_categorias = (df_calc_pag.groupby('Categoría')['inestabilidad']
                   #Sacamos el promedio de inestabilidd solo por categoría, ya que necesitamos identificar que varíen independientemente del proceso.
                   .mean() 
                   .sort_values(ascending=False)
                   .head(5)
                   .index.tolist())

df_filtrado = df_costos[df_costos['Categoría'].isin(top5_categorias)].copy()

# Centramos los datos en cero restando el promedio a cada valor costeado.
df_filtrado['variacion_neta'] = df_filtrado.groupby(['Categoría', 'Proceso Producción'])['costo_antes_iva'].transform(lambda x: x - x.mean())

# Gráfica de cajas para visualizar la variación neta de los valores costeados.
fig_cos = px.box(df_filtrado, 
             x='Categoría', 
             y='variacion_neta', 
             color='Proceso Producción',
             points='outliers',
             template='plotly_dark')

# Línea de referencia en cero
fig_cos.add_hline(y=0, line_dash="dash", line_color="pink")

fig_cos.update_layout(
    boxmode='group',
    yaxis_title="Desviación del Costo ($)",
    xaxis_title="Top 5 Categorías con Procesos más Inestables"
)
#fig_os.update_traces(boxmean='sd')
fig_cos.show()

## Visualización de datos del valor costeado por OS

In [39]:
# Agrupamos por Categoría y proceso de producción, calculando la desviación estándar y el promedio del valor costeado
df_calc_os = df_costos.groupby(['Categoría', 'Proceso Producción'])['OS-Valor Unitario'].agg(['std', 'mean']).reset_index()

df_calc_os['inestabilidad'] = df_calc_os['std'] / df_calc_os['mean']

# Utilizamos el calculo de inestabilidad para identificar las 5 categorías con procesos más inestables.
top5_categorias = (df_calc_os.groupby('Categoría')['inestabilidad']
                   #Sacamos el promedio de inestabilidd solo por categoría, ya que necesitamos identificar que varíen independientemente del proceso.
                   .mean() 
                   .sort_values(ascending=False)
                   .head(5)
                   .index.tolist())


df_filtrado = df_costos[df_costos['Categoría'].isin(top5_categorias)].copy()

# Centramos los datos: Restamos el promedio específico de cada Categoría-Proceso
# para que el "0" sea el punto de equilibrio de cada uno.
df_filtrado['variacion_netaos'] = df_filtrado.groupby(['Categoría', 'Proceso Producción'])['OS-Valor Unitario'].transform(lambda x: x - x.mean())

# --- PASO 4: Gráfica de "Inestabilidad Pura" ---
fig_os = px.box(df_filtrado, 
             x='Categoría', 
             y='variacion_netaos', 
             color='Proceso Producción',
             points='all',
             title='Inestabilidad por OS (Centrado en Cero)',
             template='plotly_dark')

# Línea de referencia en cero
fig_os.add_hline(y=0, line_dash="dash", line_color="blue")

fig_os.update_layout(
    boxmode='group',
    yaxis_title="Desviación del Costo ($)",
    xaxis_title="Top 5 Categorías con Procesos más Inestables"
)

fig_os.show()

## Visualización de la diferencia entre los costeos

In [40]:
# Agrupamos por Categoría y proceso de producción, calculando la desviación estándar y el promedio del valor costeado
df_calc_dif = df_costos.groupby(['Categoría', 'Proceso Producción'])['Diferencia valor unitario'].agg(['std', 'mean']).reset_index()

df_calc_dif['inestabilidad'] = df_calc_dif['std'] / df_calc_dif['mean']

# Utilizamos el calculo de inestabilidad para identificar las 5 categorías con procesos más inestables.
top5_categorias = (df_calc_dif.groupby('Categoría')['inestabilidad']
                   #Sacamos el promedio de inestabilidd solo por categoría, ya que necesitamos identificar que varíen independientemente del proceso.
                   .mean() 
                   .sort_values(ascending=False)
                   .head(5)
                   .index.tolist())


df_filtrado = df_costos[df_costos['Categoría'].isin(top5_categorias)].copy()

# Centramos los datos: Restamos el promedio específico de cada Categoría-Proceso
# para que el "0" sea el punto de equilibrio de cada uno.
df_filtrado['variacion_netadif'] = df_filtrado.groupby(['Categoría', 'Proceso Producción'])['Diferencia valor unitario'].transform(lambda x: x - x.mean())

# --- PASO 4: Gráfica de "Inestabilidad Pura" ---
fig_dif = px.box(df_filtrado, 
             x='Categoría', 
             y='variacion_netadif', 
             color='Proceso Producción',
             points='all',
             template='plotly_dark')

# Línea de referencia en cero
fig_dif.add_hline(y=0, line_dash="dash", line_color="blue")

fig_dif.update_layout(
    boxmode='group',
    yaxis_title="Desviación del Costo ($)",
    xaxis_title="Top 5 Categorías con Procesos más Inestables"
)

fig_dif.show()

# Valor costeado vs pagado en margen de ahorro o sobrecosto

### Confección

In [41]:
# Solo creamos la columna de estado para el color del gráfico
df_costos['estado'] = df_costos['Diferencia'].apply(
    lambda x: 'Ahorro' if x < 0 else 'Sobrecosto'
)

df_confeccion = df_costos[df_costos['Proceso Producción'] == 'Confección'].copy()

# Sumamos la diferencia por OP para que solo exista una barra por etiqueta
df_consolidado = df_confeccion.groupby(['Categoría', 'OP Det', 'estado']).agg({
    'Diferencia': 'median'
}).reset_index()

# Ahora sí calculamos el impacto absoluto sobre el total de la OP
df_consolidado['abs_impacto'] = df_consolidado['Diferencia'].abs()

# Top 10 de las OPs con mayor impacto real
top_impacto = df_consolidado.sort_values(by='abs_impacto', ascending=False).head(10)

top_impacto['Etiqueta'] = top_impacto['Categoría'] + " (OP: " + top_impacto['OP Det'].astype(str) + ")"

fig = px.bar(
    top_impacto,
    x='Diferencia',
    y='Etiqueta',
    orientation='h',
    color='estado',
    color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    template='plotly_dark'
)

# Esto elimina las líneas negras que dividen los segmentos dentro de la barra
fig.update_traces(marker_line_width=0) 

fig.show()

In [42]:
# Filtramos el DF para obtener solo los datos de confección.
df_conf = df_costos[df_costos['Proceso Producción'] == 'Confección'].copy()

# Definimos una columna de estado para tener el contexto de qué significa cada punto en la gráfica.
df_conf['estado'] = df_conf.apply(
    lambda 
    x: 'Ahorro' 
    if x['OS-Valor Unitario'] < x['costo_antes_iva'] 
    else 'Sobrecosto',
    axis=1
)

# Creamos la gráfica de confección, lo que esta encima de la línea diagonal es ahorro y lo que esta debajo es sobrecosto
# El tamaño de cada punto representa el impacto económico

fig_confe = px.scatter(
    df_conf,
    x='OS-Valor Unitario',
    y='costo_antes_iva',
    color='Categoría',
    template='plotly_dark',
    #color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    title='Confección'
)

min_val = min(df_conf['costo_antes_iva'].min(), df_conf['OS-Valor Unitario'].min())
max_val = max(df_conf['costo_antes_iva'].max(), df_conf['OS-Valor Unitario'].max())

fig_confe.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="pink", dash="dash")
)
fig_confe.update_traces(marker=dict(size=10))  # Cambia el número a tu gusto (8-12 suele quedar bien)
fig_confe.show()


### Sublimación

In [43]:
# Filtramos el DF para obtener solo los datos de sublimación.
df_subl = df_costos[df_costos['Proceso Producción'] == 'Sublimación'].copy()

# Definimos una columna de estado para tener el contexto de qué significa cada punto en la gráfica.
df_subl['estado'] = df_subl.apply(
    lambda 
    x: 'Ahorro' 
    if x['OS-Valor Unitario'] < x['costo_antes_iva'] 
    else 'Sobrecosto',
    axis=1
)

# Creamos la gráfica de sublimación, lo que esta encima de la línea diagonal es ahorro y lo que esta debajo es sobrecosto
# El tamaño de cada punto representa el impacto económico

fig_subl= px.scatter(
    df_subl,
    x='OS-Valor Unitario',
    y='costo_antes_iva',
    color='Categoría',
    template='plotly_dark',
    #color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    size=abs(df_subl['costo_antes_iva'] - df_subl['OS-Valor Unitario']),
    title='Sublimación'
)

min_val = min(df_subl['costo_antes_iva'].min(), df_subl['OS-Valor Unitario'].min())
max_val = max(df_subl['costo_antes_iva'].max(), df_subl['OS-Valor Unitario'].max())

fig_subl.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="pink", dash="dash")
)

fig_subl.show()

In [44]:
# Solo creamos la columna de estado para el color del gráfico
df_costos['estado'] = df_costos['Diferencia'].apply(
    lambda x: 'Ahorro' if x < 0 else 'Sobrecosto'
)

df_subli = df_costos[df_costos['Proceso Producción'] == 'Sublimación']

# Sumamos la diferencia por OP para que solo exista una barra por etiqueta
df_consolidado_subl = df_subli.groupby(['Categoría', 'OP Det', 'estado']).agg({
    'Diferencia': 'median'
}).reset_index()

# Ahora sí calculamos el impacto absoluto sobre el total de la OP
df_consolidado_subl['abs_impacto'] = df_consolidado_subl['Diferencia'].abs()

# Top 10 de las OPs con mayor impacto real
top_impacto_sub = df_consolidado_subl.sort_values(by='abs_impacto', ascending=False).head(10)

top_impacto_sub['Etiqueta'] = top_impacto_sub['Categoría'] + " (OP: " + top_impacto_sub['OP Det'].astype(str) + ")"

fig_sub = px.bar(
    top_impacto_sub,
    x='Diferencia',
    y='Etiqueta',
    orientation='h',
    color='estado',
    color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    template='plotly_dark'
)

# Esto elimina las líneas negras que dividen los segmentos dentro de la barra
fig_sub.update_traces(marker_line_width=0) 

fig_sub.show()

### Bordado

In [45]:
# Filtramos el DF para obtener solo los datos de Bordado.
df_bord = df_costos[df_costos['Proceso Producción'] == 'Bordado'].copy()

# Definimos una columna de estado para tener el contexto de qué significa cada punto en la gráfica.
df_bord['estado'] = df_bord.apply(
    lambda 
    x: 'Ahorro' 
    if x['OS-Valor Unitario'] < x['costo_antes_iva'] 
    else 'Sobrecosto',
    axis=1
)

# Creamos la gráfica de bordado, lo que esta encima de la línea diagonal es ahorro y lo que esta debajo es sobrecosto
# El tamaño de cada punto representa el impacto económico

fig_bord= px.scatter(
    df_bord,
    x='OS-Valor Unitario',
    y='costo_antes_iva',
    color='Categoría',
    template='plotly_dark',
    #color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    title='Bordado',
    size=abs(df_bord['costo_antes_iva'] - df_bord['OS-Valor Unitario']),
)

min_val = min(df_bord['costo_antes_iva'].min(), df_bord['OS-Valor Unitario'].min())
max_val = max(df_bord['costo_antes_iva'].max(), df_bord['OS-Valor Unitario'].max())

fig_bord.add_shape(
    type="line",
    x0=min_val, y0=min_val,
    x1=max_val, y1=max_val,
    line=dict(color="pink", dash="dash")
)

fig_bord.show()

In [58]:
# Solo creamos la columna de estado para el color del gráfico
df_costos['estado'] = df_costos['Diferencia'].apply(
    lambda x: 'Ahorro' if x < 0 else 'Sobrecosto'
)

df_bord = df_costos[df_costos['Proceso Producción'] == 'Bordado']

# Sumamos la diferencia por OP para que solo exista una barra por etiqueta
df_consolidado_bord = df_bord.groupby(['Categoría', 'OP Det', 'estado']).agg({
    'Diferencia': 'median'
}).reset_index()

# Ahora sí calculamos el impacto absoluto sobre el total de la OP
df_consolidado_bord['abs_impacto'] = df_consolidado_bord['Diferencia'].abs()

# Top 10 de las OPs con mayor impacto real
top_impacto_bord = df_consolidado_bord.sort_values(by='abs_impacto', ascending=False).head(10)

top_impacto_bord['Etiqueta'] = top_impacto_bord['Categoría'] + " (OP: " + top_impacto_bord['OP Det'].astype(str) + ")"

fig_bord = px.bar(
    top_impacto_bord,
    x='Diferencia',
    y='Etiqueta',
    orientation='h',
    color='estado',
    color_discrete_map={'Ahorro': 'green', 'Sobrecosto': 'red'},
    template='plotly_dark'
)

# Esto elimina las líneas negras que dividen los segmentos dentro de la barra
fig_bord.update_traces(marker_line_width=0) 

fig_bord.show()

# Análisis por Comercial

### Confección

In [46]:
# Filtramos por proceso de confección
df_confeccion = df_costos[df_costos['Proceso Producción'] == "Confección"]
df_confeccion['Tarea'] = df_confeccion['Categoría'] + " (" + df_confeccion['Proceso Producción'] + ")"

# Identificamos las tareas más costosas en el proceso de confección
top5_tareas = (df_confeccion.groupby('Tarea')['Diferencia valor unitario']
               .apply(lambda x: x.abs().sum())  # suma de absolutos, no absoluto de la suma
               .sort_values(ascending=False)
               .head(8)
               .index.tolist())

df_plot = df_confeccion[df_confeccion['Tarea'].isin(top5_tareas)]

# Agrupamos por tarea (categoría + proceso) y por comercial, calculamos el promedio del costeo de cada uno
df_comparativa = df_plot.groupby(['Tarea', 'Nombre del Comercial'])['Diferencia valor unitario'].mean().reset_index()
df_referencia = df_plot.groupby('Tarea')['Diferencia valor unitario'].mean().reset_index()

# Hacemos la gráfica de barras del proceso de confección
fig_conf_comercial = px.bar(df_comparativa, 
             x='Diferencia valor unitario', 
             y='Tarea', 
             color='Nombre del Comercial',
             barmode='group',
             orientation='h',
             text_auto='.0f',
             title='Confección',
             template='plotly_dark',
             color_discrete_sequence=px.colors.qualitative.Bold)

fig_conf_comercial.update_layout(xaxis_title="Costo Promedio ($)", yaxis_title="", showlegend=True)

fig_conf_comercial.show()

C:\Users\Laura\AppData\Local\Temp\ipykernel_804\1332218020.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_confeccion['Tarea'] = df_confeccion['Categoría'] + " (" + df_confeccion['Proceso Producción'] + ")"


### Bordado

In [47]:
# Filtramos por proceso de confección
df_bordado = df_costos[df_costos['Proceso Producción'] == "Bordado"]
df_bordado['Tarea'] = df_bordado['Categoría'] + " (" + df_bordado['Proceso Producción'] + ")"

# Identificamos las tareas más costosas en el proceso de confección
top_tareas_bordado = df_bordado.groupby('Tarea')['costo_antes_iva'].sum().sort_values(ascending=False).head(5).index.tolist()
df_plot_bordado = df_bordado[df_bordado['Tarea'].isin(top_tareas_bordado)]

# Agrupamos por tarea (categoría + proceso) y por comercial, calculamos el promedio del costeo de cada uno
df_comparativa_bordado = df_plot_bordado.groupby(['Tarea', 'Nombre del Comercial'])['costo_antes_iva'].median().reset_index()
df_referencia_bordado = df_plot_bordado.groupby('Tarea')['costo_antes_iva'].mean().reset_index()

# Hacemos la gráfica de barras del proceso de confección
fig_bordado_comercial = px.bar(df_comparativa_bordado, 
             x='costo_antes_iva', 
             y='Tarea', 
             color='Nombre del Comercial',
             barmode='group',
             orientation='h',
             text_auto='.0f',
             title='Bordado',
             template='plotly_dark',
             color_discrete_sequence=px.colors.qualitative.Bold)

# 6. Añadir las líneas de referencia (Promedio de la empresa para Sublimación)
for tarea in df_comparativa_bordado['Tarea'].unique():
    valor_ref = df_referencia_bordado[df_referencia_bordado['Tarea'] == tarea]['costo_antes_iva'].values[0]
    fig_bordado_comercial.add_vline(x=valor_ref, line_color='white', line_dash='dash', opacity=0.6)

fig_bordado_comercial.update_layout(xaxis_title="Costo Promedio ($)", yaxis_title="", showlegend=True)

fig_bordado_comercial.show()

C:\Users\Laura\AppData\Local\Temp\ipykernel_804\1682522420.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_bordado['Tarea'] = df_bordado['Categoría'] + " (" + df_bordado['Proceso Producción'] + ")"


### Sublimación

In [48]:
# Filtramos por proceso de sublimación
df_sublimacion = df_costos[df_costos['Proceso Producción'] == "Sublimación"]
df_sublimacion['Tarea'] = df_sublimacion['Categoría'] + " (" + df_sublimacion['Proceso Producción'] + ")"

# Identificamos las tareas más costosas en el proceso de sublimación
top_tareas_sublimacion = df_sublimacion.groupby('Tarea')['costo_antes_iva'].sum().sort_values(ascending=False).head(10).index.tolist()
df_plot_sublimacion = df_sublimacion[df_sublimacion['Tarea'].isin(top_tareas_sublimacion)]

# Agrupamos por tarea (categoría + proceso) y por comercial, calculamos el promedio del costeo de cada uno
df_comparativa_sublimacion = df_plot_sublimacion.groupby(['Tarea', 'Nombre del Comercial'])['costo_antes_iva'].median().reset_index()
df_referencia_sublimacion = df_plot_sublimacion.groupby('Tarea')['costo_antes_iva'].mean().reset_index()

# Hacemos la gráfica de barras del proceso de sublimación
fig_sublimacion_comercial = px.bar(df_comparativa_sublimacion, 
             x='costo_antes_iva', 
             y='Tarea', 
             color='Nombre del Comercial',
             barmode='group',
             orientation='h',
             text_auto='.0f',
             title='Sublimación',
             template='plotly_dark',
             color_discrete_sequence=px.colors.qualitative.Bold)

# 6. Añadir las líneas de referencia (Promedio de la empresa para Sublimación)
for tarea in df_comparativa_sublimacion['Tarea'].unique():
    valor_ref = df_referencia_sublimacion[df_referencia_sublimacion['Tarea'] == tarea]['costo_antes_iva'].values[0]
    fig_sublimacion_comercial.add_vline(x=valor_ref, line_color='white', line_dash='dash', opacity=0.6)

fig_sublimacion_comercial.update_layout(xaxis_title="Costo Promedio ($)", yaxis_title="", showlegend=True)

fig_sublimacion_comercial.show()

C:\Users\Laura\AppData\Local\Temp\ipykernel_804\3406594885.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sublimacion['Tarea'] = df_sublimacion['Categoría'] + " (" + df_sublimacion['Proceso Producción'] + ")"


# INFORME EN HTML

In [49]:
import base64

with open("logo.png", "rb") as image_file:
    logo_base64 = base64.b64encode(image_file.read()).decode()

In [50]:
import plotly
print(plotly.__version__)

6.6.0


In [51]:
import plotly.io as pio

for f_fig in [figboxplot, fig_cos, fig_os, fig_dif, fig, 
              fig_confe, fig_bord, fig_subl, fig_conf_comercial, 
              fig_bordado_comercial, fig_sublimacion_comercial, 
              figboxtotal, fig_sub]:
    f_fig.update_layout(
        autosize=True,
        height=450,
        width=None,
        margin=dict(l=0, r=0, t=50, b=0)
    )

In [ ]:
import os
from plotly.io import write_html

def export_reporte():
    output_dir = "./"
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, "index.html"), "w", encoding="utf-8") as f:

        # ── HEAD + ESTILOS ──────────────────────────────────────────
        f.write(f"""<!DOCTYPE html>
    <html lang='es'>
    <head>
    <meta charset='UTF-8'>
    <meta name='viewport' content='width=device-width, initial-scale=1.0'>
    <title>Informe de Costos</title>
    <style>
    
    /* Ocultar los radio buttons reales */
    input[type="radio"] {{
        display: none;
    }}

    /* Estilo de los "Botones" (Labels) */
                
    label {{
        display: inline-block;
        padding: 10px 25px;
        font-family: "Times New Roman", serif;
        font-size: 18px;
        background: #e0e0e0;
        cursor: pointer;
        border-radius: 10px 10px 0 0; /* Solo redondeado arriba */
        margin-right: 5px;
        transition: 0.3s;
    }}

    /* Efecto al pasar el mouse */
    label:hover {{
        background: #d0d0d0;
    }}

    /* Estilo del botón cuando está seleccionado */
    input[type="radio"]:checked + label {{
        background: #271B3F; /* Color oscuro para resaltar */
        color: white;
        font-weight: bold;
    }}

    /* Ocultar todos los contenidos por defecto */
    .tab-content {{
        display: none;
        padding: 20px;
        background: #ECE9FB;
        border-radius: 0 15px 15px 15px;
        box-shadow: 0 8px 16px rgba(0,0,0,0.1);
    }}

    /* Mostrar solo el contenido del radio button seleccionado */
    #tab-seleccion:checked ~ #content-seleccion,
    #tab-confeccion:checked ~ #content-confeccion,
    #tab-sublimacion:checked ~ #content-sublimacion,
    #tab-bordado:checked ~ #content-bordado{{
        display: block;
    }}
                
        body {{
            font-family: "Times New Roman", serif;
            margin: 40px;
            background-color: #ECE9FB;
            color: #673ded;
        }}
        .header {{
            position: relative;
            padding: 20px 0;
            border-bottom: 2px solid #ccc;
        }}
        .title {{
            font-size: 40px;
            font-weight: bold;
            color: #4A3BB1;
            text-align: center;
        }}
        .logo {{
            position: absolute;
            top: 50%;
            transform: translateY(-50%);
            height: 70px;
        }}
        .logo.left {{ left: 10px; }}
        .logo.right {{ right: 10px; }}
                
        .h2 {{
            font-size: 28px;
            color: #B8A6E3;
            font-weight: bold;
            margin-top: 50px;
        }}
        
        .row {{
            display: flex;
            gap: 30px;
            flex-wrap: wrap;
            margin-top: 30px;
        }}
                
        .card .plotly-graph-div {{
            width: 100% !important;
            min-width: 0 !important;
            border-radius: 15px; 
            overflow: hidden;
        }}
                
        .card {{
            flex: 1 1 45%;
            min-width: 400px;
            background-color: #F1EFFF;
            padding: 20px;
            width: 100%;
            border-radius: 10px;
            box-shadow: 0 10px 20px rgba(0,0,0,0.19), 0 6px 6px rgba(0,0,0,0.23);
        }}
        .card h3 {{
            text-align: center;
            font-size: 25px;
            color: #7166B0;
            font-weight: bold;
        }}
        .card p {{
            margin-top: 15px;
            font-size: 20px;
            color: #7166B0;
            font-weight: bold;
            text-align: center;
        }}
    </style>
    
    <script>
        // 1. Función maestra para redimensionar
        function ajustarGraficas() {{
            var graficas = document.querySelectorAll('.plotly-graph-div');
            graficas.forEach(function(g) {{
                Plotly.Plots.resize(g);
            }});
        }}

        // 2. Escuchar cambios en las pestañas (Radio Buttons)
        document.addEventListener('change', function(e) {{
            if (e.target.name === 'tabs') {{
                // Esperamos 50ms a que el CSS muestre el contenido antes de redimensionar
                setTimeout(ajustarGraficas, 50);
            }}
        }});

        // 3. Ejecutar al cargar la página (por seguridad)
        window.addEventListener('load', function() {{
            setTimeout(ajustarGraficas, 300);
        }});

        // 4. Ejecutar si el usuario cambia el tamaño de la ventana del navegador
        window.addEventListener('resize', ajustarGraficas);
    </script>
                
</head>
<body>

<div class="header">
    <img src="data:image/png;base64,{logo_base64}" class="logo left">
    <div class="title">INFORME DE COSTOS DE SERVICIOS MOLT</div>
    <img src="data:image/png;base64,{logo_base64}" class="logo right">
</div>
""")

        # ── FILA 1 ──────────────────────────────────────────────────
        f.write("<div class='row'>")

        f.write("<div class='card'><h3>DIFERENCIA DE COSTO UNITARIO</h3>")
        write_html(figboxplot, file=f, full_html=False,config={"responsive": True}, include_plotlyjs='cdn') 
        f.write("<p>Este gráfico muestra la diferencia del valor unitario en cada uno de los procesos de producción.</p></div>")

        f.write("</div>")
        
        # ── FILA 2 ──────────────────────────────────────────────────
        f.write("<div class='row'>")

        f.write("<div class='card'><h3>DIFERENCIA DE COSTO TOTAL</h3>")
        write_html(figboxtotal, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')  # 👈 las demás en False
        f.write("<p>Este gráfico muestra la diferencia del valor total en cada uno de los procesos de producción.</p></div>")

        f.write("</div>")

        # ── Botones ──────────────────────────────────────────────────

        f.write("""
            <div class="tabs">

            <input type="radio" name="tabs" id="tab-seleccion" checked>
            <label for="tab-seleccion">SELECCIONAR 👉</label>
                
            <input type="radio" name="tabs" id="tab-confeccion">
            <label for="tab-confeccion">CONFECCIÓN</label>

            <input type="radio" name="tabs" id="tab-sublimacion">
            <label for="tab-sublimacion">SUBLIMACIÓN</label>

            <input type="radio" name="tabs" id="tab-bordado">
            <label for="tab-bordado">BORDADO</label>

            <div class="tab-content" id="content-seleccion">
                <div style="text-align: center; padding: 50px; color: #7166B0;">
                    <h2 style="opacity: 0.5;">MOLT</h2>
                    <p>Selecciona un proceso arriba para ver los detalles.</p>
                </div>
            </div>
            
            <div class="tab-content" id="content-confeccion">
                <div class='row'>
                    <div class='card'>
                        <h3>Diferencia de costos - Confección</h3>
        """)

        write_html(fig_confe, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
                        <p>Ahorro y sobrecosto en el proceso de confección.</p>
                    </div>
                </div>
            </div>
        """)

        f.write("""
            <div class="tab-content" id="content-confeccion">
                <div class='row'>
                    <div class='card'>
                        <h3>Top 8 diferencia de valor unitario por comercial</h3>
        """)

        write_html(fig_conf_comercial, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
                        <p>En esta gráfica vemos las 8 categorías con mayor diferencia entre el valor costeado y pagado del servicio de confección y cómo se relacionan estos valores con el comercial</p>
                    </div>
                </div>
            </div>
        """)

        f.write("""
            <div class="tab-content" id="content-confeccion">
                <div class='row'>
                    <div class='card'>
                        <h3>Impacto económico en la compañía del proceso de confección</h3>
        """)

        write_html(fig, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
                        <p>En esta gráfica vemos el top 10 de las OP con mayor impacto económico para la compañía teniendo en cuenta la diferencia total entre el valor costeado y pagado.</p>
                    </div>
                </div>
            </div>
        """)
        
        f.write("""
            <div class="tab-content" id="content-sublimacion">
                <div class='row'>
                    <div class='card'>
                    <h3>Diferencia de costos - Sublimación</h3>
        """)

        write_html(fig_subl, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn') 

        f.write("""
                        <p>Ahorro y sobrecosto en el proceso de sublimación, el ahorro está sobre la línea diagonal y el sobrecosto bajo ella.</p>
                    </div>
                </div>
            </div>
        """)

        f.write("""
            <div class="tab-content" id="content-sublimacion">
                <div class='row'>
                    <div class='card'>
                        <h3>Diferencia de valor unitario por comercial</h3>
        """)

        write_html(fig_sublimacion_comercial, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
                        <p>En esta gráfica vemos las categorías con mayor diferencia entre el valor costeado y pagado del servicio de sublimación y cómo se relacionan estos valores con el comercial</p>
                    </div>
                </div>
            </div>
        """)


        f.write("""
            <div class="tab-content" id="content-sublimacion">
                <div class='row'>
                    <div class='card'>
                        <h3>Impacto económico de las diferencias de costos en la compañía del proceso de sublimación</h3>
        """)

        write_html(fig_sub, file=f, full_html=False, config={"responsive": True}, include_plotlyjs='cdn')

        f.write("""
                        <p>En esta gráfica vemos las OP con mayor impacto económico para la compañía teniendo en cuenta la diferencia total entre el valor costeado y pagado.</p>
                    </div>
                </div>
            </div>
        """)

        
        
        f.write("</body></html>")

# Reporte con el nombre 'index' para poder dubirlo en github pages directamente.
    size_mb = os.path.getsize("index.html") / 1024 / 1024
    print(f"✅ Reporte exportado — Tamaño: {size_mb:.1f} MB")

export_reporte()


✅ Reporte exportado — Tamaño: 2.5 MB
